# Stochastic Calculus from Zero · Episode 22: Martingale representation: why a hedge exists

Companion notebook for the video. Every number that appears on screen or in the narration is produced by running this notebook and read from `build/outputs.json`.

> **No data.** Every path and every world in this notebook is simulated or made up, with fixed random seeds so the numbers repeat.
>
> **The share** (Series 3's): priced at 100, 20% volatility, a 3.60% risk-free rate (continuous), a call with strike 105 expiring in a year, and an illustrative real-world drift of 8%. Made up for teaching.
>
> Educational material only, not financial advice.

1. Setup · 2. Last episode's answer · 3. The theorem · 4. The option's martingale · 5. Delta, recovered from paths · 6. Hedging in practice · 7. How fast the error shrinks · 8. Complete markets · 9. This episode's question · 10. Exercises · 11. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import itertools
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

out = {"meta": {
    "series": "Stochastic Calculus from Zero",
    "episode": 22,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}}

The building blocks used throughout the series (`episodes/_shared/stochastic.py`):

In [ ]:
# Building blocks for Stochastic Calculus from Zero, written out by hand so every step can be read. Copied verbatim
# into each notebook by make_notebook.py, so every notebook runs on its own. Needs: itertools, math, numpy as np.
#   Coin world (Part 1): a share at 100 that each toss multiplies by 1.2 (heads) or 0.8 (tails), three tosses.
#   Real-world chance of heads 0.6; a deposit grows 2% a step, so the risk-neutral chance is 0.55. Made up for
#   teaching.

class CoinWorld:
    """Every possible world of a few coin tosses, with real-world (P) and risk-neutral (Q) weights.

    A world is a string such as "HTH". Its first n letters are what you know after n tosses, so the block of
    information at time n that a world sits in is simply world[:n].
    """

    def __init__(self, s0=100.0, up=1.2, down=0.8, rate=0.02, p=0.6, steps=3):
        self.s0, self.up, self.down, self.rate, self.p, self.steps = s0, up, down, rate, p, steps
        self.q = (1 + rate - down) / (up - down)   # the odds that make the discounted share a martingale
        self.worlds = ["".join(w) for w in itertools.product("HT", repeat=steps)]

    def weight(self, world, measure="P"):
        heads = self.p if measure == "P" else self.q
        return float(np.prod([heads if c == "H" else 1 - heads for c in world]))

    def path(self, world):
        """Share prices S0, S1, ..., one per toss, along this world."""
        s = [self.s0]
        for c in world:
            s.append(s[-1] * (self.up if c == "H" else self.down))
        return s

    def price(self, world, n):
        return self.path(world)[n]

    def expect(self, f, measure="P"):
        return sum(self.weight(w, measure) * f(w) for w in self.worlds)

    def given(self, n, f, measure="P", by=None):
        """E[f | information at time n], as {block: best guess}. `by` groups worlds some other way (e.g. by price)."""
        key = by or (lambda w: w[:n])
        blocks = {}
        for w in self.worlds:
            blocks.setdefault(key(w), []).append(w)
        return {b: sum(self.weight(w, measure) * f(w) for w in ws) / sum(self.weight(w, measure) for w in ws)
                for b, ws in blocks.items()}

    def table(self, **extra):
        """One row per world (HHH ... TTT): weights, prices S0..Sn, number of heads, plus extra columns
        given as functions of the world, e.g. table(call=lambda w: max(cw.price(w, 3) - 100, 0))."""
        rows = []
        for w in self.worlds:
            row = {"world": w, "P": self.weight(w, "P"), "Q": self.weight(w, "Q"), "heads": w.count("H")}
            row |= {f"S{n}": s for n, s in enumerate(self.path(w))}
            row |= {k: f(w) for k, f in extra.items()}
            rows.append(row)
        return rows

    def blocks(self, n):
        """The blocks of information at time n: lists of worlds you cannot yet tell apart."""
        out = {}
        for w in self.worlds:
            out.setdefault(w[:n], []).append(w)
        return list(out.values())


def brownian_paths(n_paths, steps, years, rng):
    """n_paths Brownian paths on a grid of `steps` equal steps: returns (t, W, dW), W starting at 0."""
    dt = years / steps
    dW = rng.standard_normal((n_paths, steps)) * np.sqrt(dt)
    W = np.concatenate([np.zeros((n_paths, 1)), dW.cumsum(axis=1)], axis=1)
    return np.linspace(0, years, steps + 1), W, dW


def sum_of_squares(increments):
    """Sum of squared steps along each path (the quadratic variation on this grid)."""
    return (np.asarray(increments) ** 2).sum(axis=-1)


def chain_rule_sum(W, dW):
    """The school chain rule's prediction for W_T^2: add up 2 W dW, with W taken at the start of each step."""
    return 2 * (W[..., :-1] * dW).sum(axis=-1)


# ---- Part 2 additions -------------------------------------------------------------------------------------------
def riemann_sums(W, dW):
    """Sum of W dW sampled at the left end, the middle and the right end of each step (one path)."""
    return {"left": float((W[:-1] * dW).sum()), "mid": float((0.5 * (W[:-1] + W[1:]) * dW).sum()),
            "right": float((W[1:] * dW).sum())}


def ito_check(f, f1, f2, W, years=1.0):
    """For one path W on an even grid: the exact change f(W_T) - f(W_0), and what the school chain rule
    (sum of f'(W) dW) and the Ito formula (that plus 1/2 sum f''(W) dt) say it is."""
    dW, dt = np.diff(W), years / (len(W) - 1)
    chain = float((f1(W[:-1]) * dW).sum())
    return {"exact": float(f(W[-1]) - f(W[0])), "chain": chain, "ito": chain + 0.5 * float((f2(W[:-1]) * dt).sum())}


def correlated_paths(n_paths, steps, rho, rng, years=1.0):
    """Two Brownian motions with correlation rho, built from independent ones (Cholesky): returns (t, W1, W2)."""
    t, A, dA = brownian_paths(n_paths, steps, years, rng)
    _, _, dB = brownian_paths(n_paths, steps, years, rng)
    dW2 = rho * dA + np.sqrt(1 - rho ** 2) * dB
    return t, A, np.concatenate([np.zeros((n_paths, 1)), dW2.cumsum(axis=1)], axis=1)


def realised_vol(log_returns, years):
    """Annualised volatility from log returns: the square root of their sum of squares per year."""
    return np.sqrt((np.asarray(log_returns) ** 2).sum(axis=-1) / years)


# ---- Part 3 additions -------------------------------------------------------------------------------------------
#   Short rate (illustrative): r0 3.85%, long-run level 3.50%, mean reversion 0.30, volatility 1% (Vasicek).
def norm_cdf(x):
    """Standard normal distribution function, for numbers or numpy arrays."""
    return 0.5 * (1 + np.vectorize(math.erf)(np.asarray(x) / math.sqrt(2)))


def bs_call(s, k, t, r, vol):
    """Black-Scholes call price and delta (Series 3, Episode 13)."""
    d1 = (np.log(s / k) + (r + 0.5 * vol ** 2) * t) / (vol * np.sqrt(t))
    return s * norm_cdf(d1) - k * np.exp(-r * t) * norm_cdf(d1 - vol * np.sqrt(t)), norm_cdf(d1)


def ou_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of an Ornstein-Uhlenbeck (Vasicek) rate t years ahead."""
    return theta + (r0 - theta) * math.exp(-a * t), sigma * math.sqrt((1 - math.exp(-2 * a * t)) / (2 * a))


def ou_step(r, dt, a, theta, sigma, z):
    """One exact OU step of length dt from r, driven by standard normals z: no discretisation error."""
    return theta + (r - theta) * math.exp(-a * dt) + sigma * math.sqrt((1 - math.exp(-2 * a * dt)) / (2 * a)) * z


def vasicek_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price P = A exp(-B r) in the Vasicek model, tau years to maturity."""
    B = (1 - math.exp(-a * tau)) / a
    lnA = (theta - sigma ** 2 / (2 * a * a)) * (B - tau) - sigma ** 2 * B * B / (4 * a)
    return np.exp(lnA - B * np.asarray(r))


def cir_moments(t, a, theta, sigma, r0):
    """Mean and standard deviation of a CIR rate t years ahead."""
    e = math.exp(-a * t)
    var = r0 * sigma ** 2 / a * (e - e * e) + theta * sigma ** 2 / (2 * a) * (1 - e) ** 2
    return theta + (r0 - theta) * e, math.sqrt(var)


def cir_bond(tau, r, a, theta, sigma):
    """Zero-coupon bond price in the CIR model (Cox, Ingersoll and Ross, 1985)."""
    g = math.sqrt(a * a + 2 * sigma * sigma)
    den = (g + a) * (math.exp(g * tau) - 1) + 2 * g
    B = 2 * (math.exp(g * tau) - 1) / den
    A_ = (2 * g * math.exp((a + g) * tau / 2) / den) ** (2 * a * theta / sigma ** 2)
    return A_ * np.exp(-B * np.asarray(r))

## 2. Last episode's answer

$\mu = r$ gives $\theta = 0$, so $Z_T = e^0 = 1$: the real-world weights are already risk-neutral.

In [ ]:
out["prev"] = {"theta": 0, "Z": 1}

## 3. The theorem

**Martingale representation theorem.** Let $\widetilde W$ be a Brownian motion under $Q$ and $\mathcal F_t$ the information it generates. Every martingale $M$ with respect to $\mathcal F_t$ (with $E[M_T^2] < \infty$) can be written as

$$M_t = M_0 + \int_0^t H_s\,d\widetilde W_s$$

for some adapted process $H$. Part 2 showed that Itô integrals are martingales; this is the converse: there are no other martingales.

## 4. The option's martingale

Take $M_t = e^{-rt}V_t$, the discounted price of the call, $V_t = E_Q[e^{-r(T-t)}\max(S_T - K, 0) \mid \mathcal F_t]$. It is a $Q$-martingale (tower property, Part 1), so $dM = H\,d\widetilde W$ for some $H$. The discounted share is also one: $d(e^{-rt}S) = \sigma e^{-rt}S\,d\widetilde W$. Divide:

$$d(e^{-rt}V_t) = \Delta_t\,d(e^{-rt}S_t), \qquad \Delta_t = \frac{H_t}{\sigma e^{-rt}S_t}.$$

Holding $\Delta_t$ shares, financed at the risk-free rate, copies the option exactly. For Black-Scholes, Itô's formula identifies $\Delta_t = N(d_1)$.

In [ ]:
S0, VOL, R, MU, K, T = 100.0, 0.20, 0.036, 0.08, 105.0, 1.0
price0, delta0 = (float(v) for v in bs_call(S0, K, T, R, VOL))
out["inputs"] = {"price": price0, "delta0": delta0, "K": K, "mu_pct": MU * 100}
out["inputs"]

## 5. Delta, recovered from paths

The representation says that over a short step, the change in the discounted option price is $\Delta$ times the change in the discounted share price, plus nothing else of order $\sqrt{dt}$. So regress one on the other across many simulated one-day steps from today.

In [ ]:
rng = np.random.default_rng(seed)
h = 1 / 252
z = rng.standard_normal(200_000)
S1 = S0 * np.exp((R - 0.5 * VOL ** 2) * h + VOL * math.sqrt(h) * z)
dV = math.exp(-R * h) * bs_call(S1, K, T - h, R, VOL)[0] - price0
dS = math.exp(-R * h) * S1 - S0
slope = float(np.polyfit(dS, dV, 1)[0])
resid_share = float(1 - np.var(dV - slope * dS) / np.var(dV))
out["regress"] = {"slope": slope, "delta0": delta0, "r2": resid_share, "paths": len(z)}
assert abs(slope - delta0) < 0.01
out["regress"]

## 6. Hedging in practice

Sell the call for its Black-Scholes price, buy $\Delta$ shares, keep the rest in cash at 3.6%, and rebalance $n$ times over the year. The share follows the **real-world** drift of 8%. At expiry, the portfolio minus the payoff is the hedging error. Continuous rebalancing would make it zero; discrete rebalancing leaves an error that shrinks as $n$ grows.

In [ ]:
def hedge(n, paths=5_000, mu=MU, vol_hedge=VOL, rng=rng, keep=False):
    '''Short call sold at the Black-Scholes price for vol_hedge, delta-hedged n times with that volatility; the share has
    drift mu and volatility VOL. Returns final errors (and one path if keep).'''
    dt = T / n
    premium = float(bs_call(S0, K, T, R, vol_hedge)[0])
    s = np.full(paths, S0)
    d = bs_call(s, K, T, R, vol_hedge)[1]
    cash = premium - d * s
    track = [(0.0, S0, price0, premium)] if keep else None
    for i in range(n):
        s = s * np.exp((mu - 0.5 * VOL ** 2) * dt + VOL * math.sqrt(dt) * rng.standard_normal(paths))
        cash = cash * math.exp(R * dt)
        left = T - (i + 1) * dt
        if left > 1e-12:
            nd = bs_call(s, K, left, R, vol_hedge)[1]
            cash -= (nd - d) * s
            d = nd
        if keep:
            v = bs_call(s[0], K, left, R, VOL)[0] if left > 1e-12 else max(s[0] - K, 0.0)
            track.append(((i + 1) * dt, float(s[0]), float(v), float(d[0] * s[0] + cash[0])))
    err = d * s + cash - np.maximum(s - K, 0)
    return (err, track) if keep else err

rows = []
for n in (12, 52, 252):
    e = hedge(n)
    rows.append({"rebalances": n, "label": {12: "monthly", 52: "weekly", 252: "daily"}[n],
                 "error_sd": float(e.std()), "error_mean": float(e.mean()), "error_se": float(e.std() / math.sqrt(len(e)))})
hedge_ = {"rows": rows, "paths": 5_000, "monthly": rows[0]["error_sd"], "weekly": rows[1]["error_sd"], "daily": rows[2]["error_sd"]}
assert rows[0]["error_sd"] > rows[1]["error_sd"] > rows[2]["error_sd"]
assert all(abs(r["error_mean"]) < 4 * r["error_se"] + 0.02 for r in rows)
out["hedge"] = hedge_
pd.DataFrame(rows)

One path, rebalanced weekly: the option's value and the hedge portfolio's value move together.

In [ ]:
_, track = hedge(52, paths=1, rng=np.random.default_rng(7), keep=True)
tr = np.array(track)
out["one"] = {"t": tr[:, 0].round(4).tolist(), "option": tr[:, 2].round(3).tolist(), "portfolio": tr[:, 3].round(3).tolist(),
              "final_option": float(tr[-1, 2]), "final_portfolio": float(tr[-1, 3]), "final_share": float(tr[-1, 1]),
              "ticks": [{"x": x, "label": lab} for x, lab in ((0, "0"), (0.25, "3m"), (0.5, "6m"), (0.75, "9m"), (1, "1y"))]}
{k: v for k, v in out["one"].items() if k.startswith("final")}

## 7. How fast the error shrinks

Each rebalancing interval leaves an error of order $\Gamma S^2\big((\Delta W)^2 - \Delta t\big)$, mean zero and size $\Delta t$; $n$ of them add up like $\sqrt n\,\Delta t \propto 1/\sqrt n$. From monthly to daily is $252/12 = 21$ times more rebalancing, so the error should shrink by about $\sqrt{21}$.

In [ ]:
out["rate"] = {"ratio": rows[0]["error_sd"] / rows[2]["error_sd"], "sqrt": math.sqrt(252 / 12), "times": 252 / 12}
assert abs(out["rate"]["ratio"] / out["rate"]["sqrt"] - 1) < 0.25
out["rate"]

## 8. Complete markets

One source of randomness ($\widetilde W$) and one risky asset to trade: every payoff that depends on the path of $W$ can be copied, so it has exactly one price. The market is **complete**. Add a second source of randomness that no traded asset carries (a jump, or a random volatility as in Heston) and the representation needs a second asset; without one, the market is incomplete and prices are no longer unique (Part 4, and Series 3's Heston episode).

## 9. This episode's question

The error shrinks like $1/\sqrt n$: rebalancing 4 times as often halves it.

In [ ]:
e4 = hedge(208, rng=np.random.default_rng(8))
out["question"] = {"weekly": rows[1]["error_sd"], "four_times": float(e4.std()), "ratio": rows[1]["error_sd"] / float(e4.std())}
assert 1.6 < out["question"]["ratio"] < 2.4
out["question"]

## 10. Exercises

Three to do by hand, two in code. Try each before opening its solution. The code solutions run with the rest of the notebook, so their checks are tested every time.

**Exercise 1 (By hand).** Write $e^{-rT}S_T$ as $M_0 + \int_0^T H_t\,d\widetilde W_t$. What is $H$, and what is the hedge?

<details><summary>Answer</summary>

$d(e^{-rt}S_t) = \sigma e^{-rt}S_t\,d\widetilde W_t$, so $M_0 = S_0$, $H_t = \sigma e^{-rt}S_t$. The hedge is $\Delta = 1$: hold one share.

</details>

**Exercise 2 (By hand).** Why is the average hedging error close to zero even though the share drifts at 8%, not 3.6%?

<details><summary>Answer</summary>

The hedged portfolio's value changes by $\Delta\,dS - dV + r(\cdot)\,dt$; the $dW$ and $\mu\,dt$ terms cancel between $\Delta\,dS$ and $dV$ (Black-Scholes derivation). What is left over each step depends on $(\Delta W)^2 - \Delta t$, whose mean is 0 under any drift.

</details>

**Exercise 3 (By hand).** A trader rebalances monthly and sees an error spread of 2.0. Roughly how often must they rebalance to get it to 0.5?

<details><summary>Answer</summary>

Error $\propto 1/\sqrt n$: a factor of 4 smaller needs 16 times as often, $12 \times 16 = 192$ times a year (about every business day and a bit).

</details>

**Exercise 4 (In code).** Hedge (daily) with the wrong volatility, 15% instead of the true 20%. What happens to the mean error?

In [ ]:
#@title Solution 4 (double-click to show the code) { display-mode: "form" }
e_wrong = hedge(252, vol_hedge=0.15, rng=np.random.default_rng(9))
cheap = float(bs_call(S0, K, T, R, 0.15)[0])
assert e_wrong.mean() < -1
print(f"sold for {cheap:.2f} instead of {price0:.2f}; mean error {e_wrong.mean():.2f}, sd {e_wrong.std():.2f}: "
      "the call was sold too cheap for the volatility the share delivered")

**Exercise 5 (In code).** Hedge daily with real-world drift 0% and then 20%. Does the drift matter?

In [ ]:
#@title Solution 5 (double-click to show the code) { display-mode: "form" }
res = {m: hedge(252, mu=m, rng=np.random.default_rng(10)) for m in (0.0, 0.20)}
for m, e in res.items():
    print(f"drift {m:.0%}: mean error {e.mean():+.3f}, sd {e.std():.3f}")
assert abs(res[0.0].std() - res[0.20].std()) < 0.1
out["drift"] = {"zero": float(res[0.0].std()), "twenty": float(res[0.20].std())}

## 11. Export for the video

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")